# 地理阶段整理

本课把地理质量检查、代表坐标计算和地址关联串成可重复运行的流程。

| 导出文件 | 一行的含义 | 主键 |
| --- | --- | --- |
| geo_reference_v1.csv | 一个邮编前缀在一个州中的代表坐标 | 邮编前缀＋州 |
| customers_with_geo_v1.csv | 一条客户记录和对应的代表坐标或空值 | customer_id |
| sellers_with_geo_v1.csv | 一条卖家记录和对应的代表坐标或空值 | seller_id |


## 1. 读取输入和导入公共函数

地理表提供坐标来源，客户表和卖家表提供待匹配的地址。邮编前缀按字符串读取。


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from geolocation_cleaning import (
    QUALITY_FLAGS, attach_geolocation, build_geo_reference, prepare_geolocation,
    reconcile_geolocation_join, summarize_address_coverage,
)

data_dir = project_dir.parent / 'archive'
read_options = {
    'dtype': str, 'encoding': 'utf-8-sig',
    'keep_default_na': False, 'na_values': [''],
}
geo_raw = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv', **read_options)
customers_raw = pd.read_csv(data_dir / 'olist_customers_dataset.csv', **read_options)
sellers_raw = pd.read_csv(data_dir / 'olist_sellers_dataset.csv', **read_options)


## 2. 从候选记录到代表坐标

prepare_geolocation 对原始字段做整行去重，解析经纬度并加上质量标记。
build_geo_reference 以合格候选记录为输入，按邮编前缀与州分别计算经纬度中位数。
沿用第 19 课的项目粗筛矩形。 [IBGE 地理极点表](https://anuario.ibge.gov.br/images/aeb/2024/s1/2_pdf/s1t1101.pdf)


In [2]:
geo = prepare_geolocation(geo_raw)
geo_reference = build_geo_reference(geo)

display(geo[QUALITY_FLAGS].sum().rename('去重后记录数'))
print('代表坐标组数', len(geo_reference))
display(geo_reference.head())


flag_missing_or_unparsed_coordinate     0
flag_outside_world_range                0
flag_outside_brazil_screen             25
flag_invalid_zip_state                  0
Name: 去重后记录数, dtype: int64

代表坐标组数 19019


,geolocation_zip_code_prefix,geolocation_state,representative_latitude,representative_longitude,candidate_record_count,candidate_city_label_count
0,01001,SP,-23.549951,-46.634027,11,2
1,01002,SP,-23.548228,-46.635247,6,2
2,01003,SP,-23.548977,-46.635313,11,2
3,01004,SP,-23.549550,-46.634771,14,2
4,01005,SP,-23.549763,-46.636100,13,2


## 3. 关联并解释覆盖情况

attach_geolocation 按组合键做多对一左关联。
summarize_address_coverage 按邮编存在、州组合存在和候选坐标可用的顺序分类。


In [3]:
customers_with_geo = attach_geolocation(customers_raw, geo_reference, 'customer')
sellers_with_geo = attach_geolocation(sellers_raw, geo_reference, 'seller')

customer_coverage = summarize_address_coverage(customers_raw, geo, geo_reference, 'customer')
seller_coverage = summarize_address_coverage(sellers_raw, geo, geo_reference, 'seller')
address_coverage = pd.DataFrame({'客户表': customer_coverage, '卖家表': seller_coverage})
display(address_coverage)


,客户表,卖家表
地理表缺少邮编前缀,278,7
邮编前缀存在但州组合缺失,0,35
州组合存在但没有候选坐标,1,0
匹配到含候选坐标的州组合,99162,3053


## 4. 核对行数、主键与匹配数

每条左表记录在结果中占一行。分别核对原表行数、输出主键，以及匹配数与覆盖表是否一致。


In [4]:
join_controls = pd.DataFrame({
    '客户表': reconcile_geolocation_join(
        customers_raw, customers_with_geo, customer_coverage, 'customer'
    ),
    '卖家表': reconcile_geolocation_join(
        sellers_raw, sellers_with_geo, seller_coverage, 'seller'
    ),
})
display(join_controls)


,客户表,卖家表
关联前行数,99441,3095
关联后行数,99441,3095
行数差额,0,0
关联后主键重复数,0,0
匹配到代表坐标,99162,3053
未匹配到代表坐标,279,42


## 5. 导出

在项目根目录的 PowerShell 终端运行以下命令。

    python .\src\09_clean_geolocation.py

三张数据表位于 data/processed/geolocation_v1。质量检查、地址覆盖、关联核对和异常清单位于 reports/geolocation_v1。
检查是否得到 19,019 组代表坐标，客户和卖家关联的行数差额是否都为 0。

地理阶段核对完成后，整理各表的主键、字段类型和关联关系，为 MySQL 入库准备。
